# ==========================================
# PERTEMUAN 3: Missing Value, Outlier & Ekstrasi Data
## Data Cleaning - HOUSING DATASET
# ==========================================

## Import Library

In [26]:
import pandas as pd
import numpy as np
import requests
from pandas import json_normalize

## Load Dataset

In [27]:
df = pd.read_csv('housing_dirty.csv')
print('Shape awal:', df.shape)
df.head()

Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


## Eksplorasi Awal

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


Pada dataset ini terdapat **7 kolom** dengan tipe data yang beragam, yaitu **int64**, **float64**, dan **object**. Berdasarkan informasi df.info(), dataset memiliki total **130 data**. Namun, beberapa kolom seperti luas_m2, harga_juta, dan kamar masih memiliki *missing value* karena jumlah data non-null kurang dari 130.

In [29]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


Berdasarkan hasil **df.describe()**, dapat dilihat bahwa dataset memiliki statistik deskriptif seperti nilai rata-rata, minimum, maksimum, dan standar deviasi pada setiap kolom numerik. Beberapa data terlihat **tidak wajar**, seperti nilai luas_m2 dan harga_juta yang bernilai negatif, serta tahun_bangun dengan nilai maksimum 9999 yang kemungkinan merupakan data tidak valid atau **outlier**. Oleh karena itu, diperlukan proses pembersihan data agar analisis yang dilakukan menjadi lebih akurat.

In [30]:
print(df.isnull().sum())

id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


Berdasarkan hasil pengecekan missing value, terdapat beberapa kolom yang memiliki data kosong, yaitu **luas_m2 sebanyak 18 data, harga_juta sebanyak 17 data, dan kamar sebanyak 10 data**. Sementara itu, kolom lainnya tidak memiliki nilai kosong. Oleh karena itu, diperlukan penanganan missing value sebelum proses analisis atau pemodelan dilakukan.

## Find Duplicate

In [31]:
# Deteksi baris yang duplikat
n_dup = df.duplicated().sum()
print(f'{n_dup} baris duplikat dari {len(df)} total')

# Tampilkan baris yang duplikat (semua kemunculan)
df_dup = df[df.duplicated(keep=False)]

df.drop_duplicates(inplace=True)

0 baris duplikat dari 130 total


Berdasarkan hasil pengecekan data duplikat, **tidak ditemukan** baris yang memiliki data duplikat pada dataset. Hal ini ditunjukkan dengan hasil 0 baris duplikat dari 130 total sehingga kualitas data sudah cukup baik dari sisi duplikasi. Oleh karena itu, proses **drop_duplicates()** tidak menghapus data apa pun dari dataset.

## Normalisasi Sting

In [32]:
print('Isi kolom Kota')
print(df['kota'].value_counts())

Isi kolom Kota
kota
Bandung       14
Medan         13
Makassar      12
Yogyakarta    10
Jakarta        9
Depok          7
Surabaya       7
Semarang       6
medan          5
jogja          5
yogyakarta     4
makassar       4
Jogja          3
semarang       3
YGY            3
mdn            2
sby            2
dpk            2
jakarta        2
DEPOK          2
Smg            2
MAKASSAR       2
surabaya       2
Mksr           2
JAKARTA        1
Bdg            1
depok          1
bandung        1
Bandung        1
SURABAYA       1
 Jakarta       1
Name: count, dtype: int64


Berdasarkan hasil value_counts(), kolom kota memiliki beberapa penulisan yang **tidak konsisten,** seperti penggunaan huruf besar-kecil yang berbeda, singkatan, serta adanya spasi pada beberapa data. Contohnya adalah Medan, medan, dan mdn yang merujuk pada kota yang sama. Oleh karena itu, diperlukan proses data **normalisasi** dan **mapping** penulisan agar data lebih konsisten dan mudah dianalisis.

In [33]:
# Normalisasi kolom Kota
df['kota'] = df['kota'].str.strip().str.title()

mapping_kota = {
    'Bdg'      :   'Bandung',
    'Dpk'      :   'Depok',
    'Sby'      :   'Surabaya',
    'Mdn'      :   'Medan',
    'Ygy'      :   'Yogyakarta',
    'Jogja'    :   'Yogyakarta',
    'Smg'      :   'Semarang',
    'Mksr'     :   'Makassar',

}
df['kota'] = df['kota'].replace(mapping_kota)

print('Isi kolom Kota setelah normalisasi')
print(df['kota'].value_counts())

Isi kolom Kota setelah normalisasi
kota
Yogyakarta    25
Medan         20
Makassar      20
Bandung       17
Jakarta       13
Depok         12
Surabaya      12
Semarang      11
Name: count, dtype: int64


Setelah dilakukan normalisasi pada kolom `kota`, penulisan data menjadi lebih konsisten dengan menghapus spasi, menyamakan format huruf, serta mengubah singkatan menjadi nama kota yang sesuai. Hasilnya, jumlah kategori kota menjadi lebih rapi dan tidak ada lagi data duplikat akibat perbedaan penulisan. Kota dengan jumlah data terbanyak setelah normalisasi adalah `Yogyakarta`, diikuti oleh `Medan` dan `Makassar`.


In [34]:
print('Isi kolom Kondisi')
print(df['kondisi'].value_counts())

Isi kolom Kondisi
kondisi
baik              45
sedang            30
rusak             10
BAIK               8
baik sekali        7
Baik               5
SEDANG             4
cukup              4
bagus              4
Bagus              3
Sedang             3
Cukup              2
jelek              2
RUSAK              2
perlu renovasi     1
Name: count, dtype: int64


Berdasarkan hasil `value_counts()`, kolom `kondisi` masih memiliki penulisan yang **tidak konsisten**, seperti perbedaan huruf kapital serta penggunaan istilah yang berbeda untuk makna yang serupa. Contohnya adalah `baik`, `BAIK`, dan `Baik` yang seharusnya termasuk dalam kategori yang sama. Oleh karena itu, diperlukan proses normalisasi agar data kondisi menjadi lebih seragam dan mudah dianalisis.

In [35]:
# Normalisasi kolom Kondisi
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('Isi kolom Kondisi setelah normalisasi')
print(df['kondisi'].value_counts())

Isi kolom Kondisi setelah normalisasi
kondisi
baik              58
sedang            37
rusak             12
bagus              7
baik sekali        7
cukup              6
jelek              2
perlu renovasi     1
Name: count, dtype: int64


Setelah dilakukan normalisasi pada kolom `kondisi`, penulisan data menjadi lebih konsisten dengan mengubah seluruh teks menjadi huruf kecil dan menghapus spasi yang tidak diperlukan. Hasilnya, kategori seperti `BAIK` dan `Baik` berhasil digabung menjadi `baik`. Dengan demikian, data kondisi menjadi lebih rapi dan siap digunakan untuk proses analisis selanjutnya.

## Imputasi Missing Values

In [36]:
print(df.isnull().sum())

id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


Setelah proses normalisasi data, masih terdapat beberapa *missing value* pada dataset. Kolom `luas_m2` memiliki 18 data kosong, `harga_juta` sebanyak 17 data kosong, dan `kamar` sebanyak 10 data kosong. Oleh karena itu, tahap penanganan missing value tetap diperlukan sebelum dilakukan analisis atau pemodelan data.


In [37]:
# Imputasi missing values pada kolom Luas_m2, Harga_juta, dan Kamar
df['luas_m2']     = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta']  = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar']       = df['kamar'].fillna(df['kamar'].mode()[0])

print(df.isnull().sum())

id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


Setelah dilakukan proses imputasi, seluruh **missing value** pada dataset berhasil ditangani. Kolom luas_m2 dan harga_juta diisi menggunakan nilai median, sedangkan kolom kamar diisi menggunakan nilai modus. Hasilnya, semua kolom kini memiliki jumlah data lengkap tanpa nilai kosong.

## Handle Outlier (IQR Fence)

In [38]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,130.000000,1.300000e+02,130.000000,130.000000
mean,65.500000,257.405385,7.699047e+05,3.246154,2062.638462
std,37.671829,821.951924,8.770521e+06,1.826003,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,101.600000,3.805000e+02,1.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,3.000000,2002.000000
75%,97.750000,266.150000,9.160000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


Berdasarkan hasil df.describe() setelah imputasi, seluruh kolom numerik kini memiliki jumlah data lengkap sebanyak **130 data**. Namun, masih terdapat beberapa nilai yang tidak wajar, seperti luas_m2 dan harga_juta yang bernilai negatif serta tahun_bangun dengan nilai maksimum 9999 yang mengindikasikan adanya **outlier** atau data tidak valid. Oleh karena itu, diperlukan proses penanganan outlier agar kualitas data menjadi lebih baik sebelum dianalisis lebih lanjut.

In [39]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,130.000000,130.000000,130.000000,130.000000
mean,65.500000,188.274423,686.490385,3.246154,2001.542308
std,37.671829,95.297150,404.633957,1.826003,13.505818
min,1.000000,-50.000000,-422.750000,1.000000,1960.500000
25%,33.250000,101.600000,380.500000,1.000000,1991.250000
50%,65.500000,193.800000,655.000000,3.000000,2002.000000
75%,97.750000,266.150000,916.000000,5.000000,2011.750000
max,130.000000,512.975000,1719.250000,6.000000,2042.500000


Setelah dilakukan penanganan outlier menggunakan metode IQR, nilai-nilai ekstrem pada **kolom harga_juta, luas_m2, dan tahun_bangun** berhasil dibatasi ke rentang yang lebih wajar. Hal ini terlihat dari penurunan nilai maksimum yang sebelumnya sangat tinggi, seperti tahun_bangun dari 9999 menjadi 2042.5 dan harga_juta dari 100000000 menjadi 1719.25. Dengan demikian, distribusi data menjadi lebih stabil dan lebih siap digunakan untuk analisis maupun pemodelan.

## Validasi Akhir

In [40]:
missing   = df.isnull().sum().sum()
duplicate = df.duplicated().sum()

assert missing == 0, 'Masih ada missing!'
assert duplicate == 0, 'Masih ada duplikat!'

print('Jumlah missing:', missing)
print('Jumlah duplikat:', duplicate)
print('Shape akhir:', df.shape)

Jumlah missing: 0
Jumlah duplikat: 0
Shape akhir: (130, 7)


Berdasarkan hasil validasi akhir, dataset sudah tidak memiliki missing value maupun data duplikat. Dataset akhir memiliki ukuran sebanyak 130 baris dan 7 kolom sehingga siap digunakan untuk tahap analisis atau pemodelan selanjutnya.

## Ekspor Dataset

In [41]:
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Dataset bersih tersimpan!


Dataset yang sudah clean telah di ekspor dan siap untuk pemakaian selanjutnya.

## Access API JSONPlaceholder

In [42]:
URL = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(URL, timeout=10)

    if response.status_code == 200:
        data = response.json()
        df_users = json_normalize(data, sep='_')
        print(f'Berhasil! Total data: {len(df_users)} users')
        print(f'Kolom tersedia: {df_users.columns.tolist()}')
    else:
        print(f'Error: {response.status_code}')

except requests.exceptions.ConnectionError:
    print('Gagal terhubung ke API. Cek koneksi internet!')
except requests.exceptions.Timeout:
    print('Request timeout. Coba lagi!')

Berhasil! Total data: 10 users
Kolom tersedia: ['id', 'name', 'username', 'email', 'phone', 'website', 'address_street', 'address_suite', 'address_city', 'address_zipcode', 'address_geo_lat', 'address_geo_lng', 'company_name', 'company_catchPhrase', 'company_bs']


Berdasarkan hasil pengambilan data dari API, proses request berhasil dilakukan dengan status kode `200`. Data yang diperoleh berjumlah 10 pengguna (*users*) dan berhasil dikonversi ke dalam bentuk DataFrame menggunakan `json_normalize()`. Selain itu, dataset memiliki beberapa kolom informasi seperti identitas pengguna, kontak, alamat, hingga informasi perusahaan yang siap digunakan untuk analisis lebih lanjut.

In [43]:
print('Data Users dari JSONPlaceholder API')
df_users[['id', 'name', 'email', 'address_city', 'company_name']]

Data Users dari JSONPlaceholder API


,id,name,email,address_city,company_name
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,Considine-Lockman
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth,Johns Group
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview,Abernathy Group
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,Yost and Sons
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury,Hoeger LLC


Data dari JSONPlaceholder API berhasil ditampilkan dalam bentuk tabel yang berisi informasi penting pengguna, seperti id, name, email, address_city, dan company_name. Dari data tersebut dapat dilihat bahwa setiap pengguna memiliki informasi identitas, alamat kota, serta perusahaan yang berbeda-beda sehingga dataset cukup representatif untuk latihan pengolahan data API.

In [44]:
URL_POSTS = "https://jsonplaceholder.typicode.com/posts"
params    = {'userId': 1}

try:
    response_posts = requests.get(URL_POSTS, params=params, timeout=10)

    if response_posts.status_code == 200:
        posts_data = response_posts.json()
        df_posts   = pd.DataFrame(posts_data)
        print(f'Posts milik userId=1: {len(df_posts)} post')
        print(df_posts[['id', 'userId', 'title']].head())
    else:
        print(f'Error: {response_posts.status_code}')

except requests.exceptions.ConnectionError:
    print('Gagal terhubung ke API.')

Posts milik userId=1: 10 post
   id  userId                                              title
0   1       1  sunt aut facere repellat provident occaecati e...
1   2       1                                       qui est esse
2   3       1  ea molestias quasi exercitationem repellat qui...
3   4       1                               eum et est occaecati
4   5       1                                 nesciunt quas odio


Berdasarkan hasil request ke endpoint `posts`, berhasil diperoleh 10 data postingan milik pengguna dengan `userId = 1`. Data yang ditampilkan meliputi `id`, `userId`, dan `title`, sehingga dapat digunakan untuk melihat daftar postingan yang dimiliki oleh pengguna tersebut.


## Conclusion

Berdasarkan proses analisis dan data preprocessing yang telah dilakukan, dataset berhasil **dibersihkan** dari permasalahan seperti **missing value, data duplikat, penulisan kategori yang tidak konsisten, serta outlier pada beberapa kolom numerik.** Setelah dilakukan normalisasi dan imputasi, data menjadi lebih rapi, konsisten, dan siap digunakan untuk tahap analisis maupun pemodelan lebih lanjut. Selain itu, proses pengambilan data dari API menggunakan JSONPlaceholder juga berhasil dilakukan, sehingga menunjukkan bahwa dapat diakses.